In [1]:
import pandas as pd
import numpy as np

In [17]:
# === PARAMETERS ===
customer_file = r"C:\Users\mmackenzie\OneDrive - ZELEROS GLOBAL S.L\Zeleros - Zeleros\Operaciones\4- E-drive\05- Projects\BMS\State Of Art\Synthetic data\simulation_cases.xlsx"
output_file = r"C:\Users\mmackenzie\OneDrive - ZELEROS GLOBAL S.L\Zeleros - Zeleros\Operaciones\4- E-drive\05- Projects\BMS\State Of Art\Synthetic data\test.xlsx"

# === READ CUSTOMER DEFINITIONS ===
customers = pd.read_excel(customer_file, sheet_name="Customer definitions")
customers

,Customer,Average ambient temperature,Cycles per year,Max charging SOC,Min discharging SOC
0,1,0,300,100,0
1,2,10,300,100,0
2,3,20,300,100,0
3,4,30,300,100,0
4,5,0,100,100,0
5,6,10,100,100,0
6,7,20,100,100,0
7,8,30,100,100,0
8,9,0,200,90,10
9,10,10,200,90,10


In [32]:
# === FUNCTION TO GENERATE CASES FOR EACH CUSTOMER ===
def generate_cases(customer_id, avg_temp, cycles_per_year, max_soc, min_soc):
    n_steps = int(cycles_per_year * 2)  # discharge + charge per cycle
    steps = np.arange(1, n_steps + 1)
    cases = np.array(["Discharge" if i % 2 else "Charge" for i in range(1, n_steps + 1)])

    # --- Randomized starting temperature ±10°C ---
    start_temp = np.random.uniform(avg_temp - 10, avg_temp + 10, size=n_steps)
    start_temp = np.clip(start_temp, -20, 60).round()

    start_soc = np.zeros(n_steps)
    end_soc = np.zeros(n_steps)

    # --- SOC transitions ---
    for k in range(n_steps):
        if k == 0:
            start_soc[k] = max_soc
            end_soc[k] = np.random.uniform(min_soc, min_soc + 10)
        else:
            start_soc[k] = end_soc[k - 1]
            if cases[k] == "Discharge":
                end_soc[k] = np.random.uniform(min_soc, min_soc + 10)
            else:
                end_soc[k] = np.random.uniform(max_soc - 10, max_soc)
    
    start_soc = np.clip(start_soc, 0, 100)
    end_soc = np.clip(end_soc, 0, 100)

    # --- SOH degradation model ---
    # Base fade: 10% at 25°C and 300 cycles with 100% SOC window (0–100%)
    soc_window = max_soc - min_soc
    temp_factor = 1.0 + (avg_temp - 25) * 0.02     # +2% per °C above 25
    cycle_factor = cycles_per_year / 300.0
    window_factor = 100.0 / soc_window if soc_window > 0 else 1.0  # smaller window → slower fade

    total_fade = 10.0 * temp_factor * cycle_factor / window_factor
    soh = 100.0 - (total_fade * (steps - 1) / (n_steps - 1))
    soh = np.clip(soh, 0, 100)

    # --- SOC imbalance ---
    # Starts small and randomizes with increasing cycle number
    soc_imbalance = np.random.uniform(0, 1, size=n_steps) + (steps / n_steps) * 4  # increases up to ~5%
    soc_imbalance = np.clip(soc_imbalance, 0, 5)

    # --- SOH spread ---
    # Linearly increases from 0 to 2% across steps
    soh_spread = 2.0 * (steps - 1) / (n_steps - 1)

    # --- Build dataframe ---
    df = pd.DataFrame({
        "Step": steps,
        "Case": cases,
        "StartTemp_C": np.round(start_temp, 0),
        "StartSOC_avg": np.round(start_soc, 0),
        "EndSOC_avg": np.round(end_soc, 0),
        "SOC_imbalance": np.round(soc_imbalance, 0),
        "SOH_avg": np.round(soh, 1),
        "SOH_spread": np.round(soh_spread, 1),
        "Customer_ID": customer_id,
        "CyclesPerYear": cycles_per_year,
        "MaxChargingSOC": max_soc,
        "MinDischargingSOC": min_soc,
        "SOC_Window": soc_window,
        "AvgTemp_C": avg_temp
    })
    return df

In [33]:
# === GENERATE SHEETS FOR ALL CUSTOMERS ===
writer = pd.ExcelWriter(output_file)

for _, row in customers.iterrows():
    df_customer = generate_cases(
        customer_id=row["Customer"],
        avg_temp=row["Average ambient temperature"],
        cycles_per_year=row["Cycles per year"],
        max_soc=row["Max charging SOC"],
        min_soc=row["Min discharging SOC"],
    )
    sheet_name = f"Customer_{int(row['Customer'])}"
    df_customer.to_excel(writer, sheet_name=sheet_name, index=False)

writer.close()
print("Simulation tables created successfully")

Simulation tables created successfully
